# Tabular EDA + Preprocessing Pipeline (Models 1, 3, 4)

This notebook wraps the shared preprocessing utilities in `ml_models/data/preprocessing.py` and performs:

- EDA (distributions, correlations, category counts)
- Cleaning (NaNs, outliers)
- Kerala-specific augmentation (for the crop and yield data)
- Saving processed CSVs for training scripts / other notebooks.


In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn

from google.colab import drive
drive.mount('/content/drive')

MODEL_SAVE_PATH = '/content/drive/MyDrive/krishi_mitra_models/'
DATA_PATH = '/content/drive/MyDrive/krishi_mitra_data/'

import os
from pathlib import Path
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

%load_ext autoreload
%autoreload 2

## 1) Import shared preprocessing helpers

When running from the repo root (uploaded into Colab), you can import directly from `ml_models.data.preprocessing`.

If you only copied this notebook, the cell uses `importlib` to load the module by file path.


In [ ]:
import importlib.util
from pathlib import Path

preproc_path = Path('ml_models/data/preprocessing.py')
spec = importlib.util.spec_from_file_location('km_preprocessing', preproc_path)
km = importlib.util.module_from_spec(spec)
spec.loader.exec_module(km)

km

## 2) Crop Recommendation — EDA + preprocessing

- Input: Kaggle `Crop_recommendation.csv`
- Output: Kerala-augmented train/test CSVs under `DATA_PATH/processed/`


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

raw_dir = Path(DATA_PATH) / 'raw'
proc_dir = Path(DATA_PATH) / 'processed'
proc_dir.mkdir(parents=True, exist_ok=True)

crop_csv = next(p for p in raw_dir.glob('**/Crop_recommendation.csv'))
df_raw = pd.read_csv(crop_csv)
df_raw.head()

In [ ]:
# Numeric summary + missing values
km.summarize_numeric(df_raw)

In [ ]:
# Quick distributions for core numeric features
plt.figure(figsize=(10, 6))
for i, col in enumerate(['N','P','K','temperature','humidity','ph','rainfall'], start=1):
    plt.subplot(3, 3, i)
    sns.histplot(df_raw[col], kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Full preprocessing + Kerala augmentation + train/test split
split = km.preprocess_crop_recommendation(crop_csv, proc_dir / 'crop_recommendation')
split.train.head()

## 3) Yield data — EDA + preprocessing

- Input: Kaggle crop yield CSV (largest/yield-focused CSV in `raw/`)
- Output: `yield_kerala_processed.csv` with engineered features.


In [ ]:
yield_csvs = [p for p in raw_dir.glob('*.csv')]
yield_csv = max(yield_csvs, key=lambda p: p.stat().st_size)
yield_csv

In [ ]:
df_yield_raw = pd.read_csv(yield_csv)
df_yield_raw.head()

In [ ]:
km.summarize_numeric(df_yield_raw)

In [ ]:
df_yield = km.preprocess_yield_data(yield_csv, proc_dir / 'yield')
df_yield[['district','crop_name','year','area_hectares','production_tons_per_hectare']].head()

In [ ]:
# Correlation heatmap (numeric features only)
import numpy as np
num_cols = df_yield.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df_yield[num_cols].corr(), cmap='coolwarm', center=0)
plt.title('Yield numeric feature correlations')
plt.show()

## 4) Pest risk — synthetic dataset

For Model 4, we rely on synthetic data (no public dataset). This cell generates and saves it.


In [ ]:
df_pest = km.preprocess_pest_risk(proc_dir / 'pest_risk', n_rows=10_000)
df_pest.head()